## Project Description: Next Word Prediction Using LSTM
#### Project Overview:

This project aims to develop a deep learning model for predicting the next word in a given sequence of words. The model is built using Long Short-Term Memory (LSTM) networks, which are well-suited for sequence prediction tasks. The project includes the following steps:

1- Data Collection: We use the text of Shakespeare's "Hamlet" as our dataset. This rich, complex text provides a good challenge for our model.

2- Data Preprocessing: The text data is tokenized, converted into sequences, and padded to ensure uniform input lengths. The sequences are then split into training and testing sets.

3- Model Building: An LSTM model is constructed with an embedding layer, two LSTM layers, and a dense output layer with a softmax activation function to predict the probability of the next word.

4- Model Training: The model is trained using the prepared sequences, with early stopping implemented to prevent overfitting. Early stopping monitors the validation loss and stops training when the loss stops improving.

5- Model Evaluation: The model is evaluated using a set of example sentences to test its ability to predict the next word accurately.

6- Deployment: A Streamlit web application is developed to allow users to input a sequence of words and get the predicted next word in real-time.

In [ ]:
import nltk
nltk.download("gutenberg")
from nltk.corpus import gutenberg
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from keras._tf_keras.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import LSTM,Dense, Embedding, Dropout, GRU
from keras.callbacks import EarlyStopping
import pickle

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [3]:
data = gutenberg.raw('shakespeare-hamlet.txt')
## Save file
with open('hamlet.txt','w') as f:
    f.write(data)

In [4]:
with open('hamlet.txt','r') as f:
    text = f.read().lower()

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
length_size = len(tokenizer.word_index) + 1
print(f"Length of Vocab - {length_size}")


Length of Vocab - 4818


In [5]:
## Creating Input sequence to train model
input_sequence= []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    # print(token_list)
    for i in range(1,len(token_list)):
        n_gram_seq = token_list[:i+1]
        # print(n_gram_seq)
        input_sequence.append(n_gram_seq)



In [6]:
# # Alternative for above 
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("gpt2")
# tokens = tokenizer(text, return_tensors="pt")


In [7]:
## Padding the Sequence
max_length_seq = max([len(i) for i in input_sequence])
max_length_seq

14

In [8]:
input_sequences = np.array(pad_sequences(input_sequence,maxlen = max_length_seq, padding='pre'))
input_sequences

array([[   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       [   0,    0,    0, ...,  687,    4,   45],
       ...,
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4],
       [   0,    0,    0, ..., 1047,    4,  193]],
      shape=(25732, 14), dtype=int32)

In [9]:
#Now Lets creaates predictors and labels

x,y = input_sequences[:,:-1],input_sequences[:,-1]
x

array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,  687],
       [   0,    0,    0, ...,    1,  687,    4],
       ...,
       [   0,    0,    0, ...,  687,    4,   45],
       [   0,    0,    0, ...,    4,   45, 1047],
       [   0,    0,    0, ...,   45, 1047,    4]],
      shape=(25732, 13), dtype=int32)

In [10]:
y

array([ 687,    4,   45, ..., 1047,    4,  193],
      shape=(25732,), dtype=int32)

In [11]:
# Multiple Words could have been repeated, so lets convert them into categories of features
y = to_categorical(y,num_classes=length_size)
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(25732, 4818))

In [12]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2)

In [ ]:
model = Sequential()
model.add(Embedding(
    input_dim=length_size, #vocab size
    output_dim=100, #embedding dimension
    input_shape=(max_length_seq-1,) #because we have removed last word for label
))
model.add(LSTM(150,return_sequences=True)) #return sequences true because we have another LSTM layer 
# Can also use GRU layer instead of LSTM
model.add(Dropout(0.2)) #Dropout to avoid overfitting
model.add(LSTM(100)) # Second LSTM layer model.add(GRU(100)) # Can also use GRU layer instead of LSTM
model.add(Dense(length_size,activation='softmax'))
model.compile(optimizer='adam',loss='categorical_crossentropy', metrics=['accuracy']) #Using categorical crossentropy because multiple words could be repeated
model.summary()

d:\Projects\Lstm Projects\vlstm\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 13, 100)        │       481,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 13, 150)        │       150,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 13, 150)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4818)           │       486,618 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,219,418 (4.65 MB)

 Trainable params: 1,219,418 (4.65 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
history = model.fit(x_train,y_train,epochs=50, verbose=8, validation_data=(x_test,y_test),callbacks=[EarlyStopping(monitor='val_accuracy',patience=5,restore_best_weights=True)])

Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.0326 - loss: 6.8751 - val_accuracy: 0.0274 - val_loss: 6.7873
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.0390 - loss: 6.4452 - val_accuracy: 0.0418 - val_loss: 6.8763
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.0462 - loss: 6.3079 - val_accuracy: 0.0488 - val_loss: 6.9278
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.0519 - loss: 6.1696 - val_accuracy: 0.0474 - val_loss: 6.9518
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.0527 - loss: 6.0460 - val_accuracy: 0.0499 - val_loss: 7.0123
Epoch 6/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.0604 - loss: 5.9140 - val_accuracy: 0.0554 - val_loss: 7.0667
Epoch 7/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - accuracy: 0.0677 - loss: 5.7735 - val_accuracy: 0.0604 - val_loss: 7.0975
Epoch 8/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - accuracy: 0.0798 - loss: 5.6247 - val_ac

In [22]:
def predic_next_word(model,text,tokenizer,max_seq_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) > max_seq_len:
        token_list = token_list[-(max_seq_len):]
    pad_token_list = pad_sequences([token_list],padding='pre',maxlen = (max_seq_len-1))
    predicted = model.predict(pad_token_list,verbose = 1)
    predicted_word_index = np.argmax(predicted,axis=1)
    for word,index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None



In [16]:
model.input_shape[1]

13

In [17]:
text = "To be or not to be"
predic_next_word(model= model, text=text, tokenizer= tokenizer, max_seq_len= model.input_shape[1]+1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 567ms/step


'much'

In [18]:
model.save('next_word_predictor_lstm.h5')

In [21]:
with open("Tokenizer.pickle",'wb') as handle:
    pickle.dump(tokenizer,handle,protocol=pickle.HIGHEST_PROTOCOL)